### Exploratory notebook for cell-count.csv dataset

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("cell-count.csv")

In [4]:
df.head()

,project,subject,condition,age,sex,treatment,response,sample,sample_type,time_from_treatment_start,b_cell,cd8_t_cell,cd4_t_cell,nk_cell,monocyte
0,prj1,sbj000,melanoma,57,M,miraclib,no,sample00000,PBMC,0,10908,24440,20491,13864,23511
1,prj1,sbj000,melanoma,57,M,miraclib,no,sample00001,PBMC,7,6777,19407,33459,18170,23011
2,prj1,sbj000,melanoma,57,M,miraclib,no,sample00002,PBMC,14,9794,22940,24274,17482,18332
3,prj1,sbj001,carcinoma,68,M,miraclib,yes,sample00003,PBMC,0,10081,20271,36157,14041,12610
4,prj1,sbj001,carcinoma,68,M,miraclib,yes,sample00004,PBMC,7,4372,33778,38293,16527,14933


In [41]:
len(df)

10500

In [35]:
df['project'].unique()

array(['prj1', 'prj2', 'prj3'], dtype=object)

In [ ]:
# num. subjects
len(df['subject'].unique())

3500

In [40]:
# num. samples
len(df['sample'].unique())

10500

Plan for DB schema: one table for samples (metadata), one table for cell counts (actual measurements)

Sample columns: sample id (pk), project, subject, condition, etc.

Cell count columns: sample id (fk), cell population, count

In [ ]:
# samples should include all metadata and exclude information about cell population
sample_cols = [
    "sample",
    "project",
    "subject",
    "condition",
    "age",
    "sex",
    "treatment",
    "response",
    "sample_type",
    "time_from_treatment_start"]

samples_df = df[sample_cols]
samples_df.head()

,sample,project,subject,condition,age,sex,treatment,response,sample_type,time_from_treatment_start
0,sample00000,prj1,sbj000,melanoma,57,M,miraclib,no,PBMC,0
1,sample00001,prj1,sbj000,melanoma,57,M,miraclib,no,PBMC,7
2,sample00002,prj1,sbj000,melanoma,57,M,miraclib,no,PBMC,14
3,sample00003,prj1,sbj001,carcinoma,68,M,miraclib,yes,PBMC,0
4,sample00004,prj1,sbj001,carcinoma,68,M,miraclib,yes,PBMC,7


In [ ]:
# all cell populations
pop_cols = [
    "b_cell",
    "cd8_t_cell",
    "cd4_t_cell",
    "nk_cell",
    "monocyte"
]

# conv. to long format
# ex row: sample_id: sample00000, population: b_cell, count: 4000
pop_df = df.melt(
    id_vars=["sample"],
    value_vars=pop_cols, # measured var
    var_name="population",
    value_name="count"
)
pop_df.head()

,sample,population,count
0,sample00000,b_cell,10908
1,sample00001,b_cell,6777
2,sample00002,b_cell,9794
3,sample00003,b_cell,10081
4,sample00004,b_cell,4372


In [ ]:
# responding melanoma males at baseline
melanoma_males = df[(df['condition']=='melanoma') 
                            & (df['sex']=='M')
                            & (df['time_from_treatment_start']==0)
                            & (df['response']=='yes')]

# mean b_cell count
print(round(float(melanoma_males['b_cell'].mean()), 2))

10206.15


In [ ]:
# count on the y, relative cell pop frequency on the x

